# Data Cleaning & Canonicalization

Transform the raw JSON records into the finalized canonical product schema.

In [1]:
!git clone https://github.com/jnDhruv/multi-modal-fashion-ecom
!ls

Cloning into 'multi-modal-fashion-ecom'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 82 (delta 26), reused 54 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 925.81 KiB | 16.24 MiB/s, done.
Resolving deltas: 100% (26/26), done.
multi-modal-fashion-ecom  __notebook__.ipynb


In [2]:
import sys
from pathlib import Path
import importlib

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/kaggle/working/multi-modal-fashion-ecom")
SCRIPTS_DIR = PROJECT_ROOT / "data" / "scripts"

sys.path.append(str(SCRIPTS_DIR))

In [3]:
!pip install -q -r /kaggle/working/multi-modal-fashion-ecom/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00


In [4]:
import pandas
import ftfy
import bs4
import pyarrow

print("Dependencies OK")

Dependencies OK


In [5]:
import clean_data

print(clean_data.__file__)

/kaggle/working/multi-modal-fashion-ecom/data/scripts/clean_data.py


In [6]:
DATASET_DIR = Path(
    "/kaggle/input/datasets/paramaggarwal/"
    "fashion-product-images-dataset/fashion-dataset"
)

STYLES_CSV = DATASET_DIR / "styles.csv"

valid_ids = clean_data.load_styles_csv_ids(STYLES_CSV)

print("Valid IDs:", len(valid_ids))
print("Sample IDs:", list(valid_ids)[:10])

Valid IDs: 44424
Sample IDs: [1163, 1164, 1165, 1525, 1526, 1528, 1529, 1530, 1531, 1532]


In [7]:
STYLES_DIR = DATASET_DIR / "styles"

json_records = clean_data.load_json_records(STYLES_DIR)

print("Loaded JSON records:", len(json_records))

Loaded JSON records: 44446


In [8]:
clean_records = clean_data.filter_valid_records(
    json_records,
    valid_ids
)

print("Original JSON records:", len(json_records))
print("Clean records:", len(clean_records))
print("Dropped:", len(json_records) - len(clean_records))

Original JSON records: 44446
Clean records: 44424
Dropped: 22


In [9]:
clean_ids = {
    record["data"]["id"]
    for record in clean_records
}

print("Unique clean IDs:", len(clean_ids))
print("All IDs valid:", clean_ids.issubset(valid_ids))
print("Missing valid IDs:", len(valid_ids - clean_ids))

Unique clean IDs: 44424
All IDs valid: True
Missing valid IDs: 0


In [10]:
# Test extract_type_name
print("1. extract_type_name")
print(clean_data.extract_type_name({
    "id": 90,
    "typeName": "Tshirts"
}))
print(clean_data.extract_type_name("Men"))
print(clean_data.extract_type_name(None))


# Test missing attribute detection
print("\n2. is_missing_attribute")
for value in [None, "", "NA", "N/A", "Regular Fit", "Cotton"]:
    print(repr(value), "->", clean_data.is_missing_attribute(value))


# Test descriptor unwrapping
print("\n3. unwrap_descriptor")
sample_descriptor = {
    "description": {
        "descriptorType": "description",
        "value": "<p>Test <strong>shirt</strong></p>"
    }
}
print(
    clean_data.unwrap_descriptor(
        sample_descriptor,
        "description"
    )
)


# Test encoding
print("\n4. fix_encoding")
print(clean_data.fix_encoding("30Â° C machine wash"))
print(clean_data.fix_encoding('32â€�'))


# Test HTML stripping
print("\n5. strip_html")
print(
    clean_data.strip_html(
        "<p>Regular <strong>fit</strong><br>cotton shirt</p>"
    )
)


# Test truncation
print("\n6. truncate_text")
long_text = "This is a test sentence " * 100
truncated = clean_data.truncate_text(long_text)

print("Original length:", len(long_text))
print("Truncated length:", len(truncated))
print("Ends cleanly:", not truncated.endswith(" "))


# Test complete description cleaning
print("\n7. clean_description")
raw_description = """
<div>
    <strong>Style Note</strong><br>
    This is a <b>30Â° C</b> cotton shirt.
</div>
"""

print(clean_data.clean_description(raw_description))

1. extract_type_name
Tshirts
Men
None

2. is_missing_attribute
None -> True
'' -> True
'NA' -> True
'N/A' -> True
'Regular Fit' -> False
'Cotton' -> False

3. unwrap_descriptor
<p>Test <strong>shirt</strong></p>

4. fix_encoding
30° C machine wash
32�

5. strip_html
Regular fit cotton shirt

6. truncate_text
Original length: 2400
Truncated length: 1497
Ends cleanly: True

7. clean_description
Style Note This is a 30° C cotton shirt.


In [11]:
sample_row = clean_data.build_row(clean_records[1000])

sample_row

{'id': 26233,
 'product_display_name': 'Proline Men Red Polo T-shirt',
 'brand_name': 'Proline',
 'gender': 'Men',
 'master_category': 'Apparel',
 'sub_category': 'Topwear',
 'article_type': 'Tshirts',
 'base_colour': 'Red',
 'season': 'Summer',
 'usage': 'Casual',
 'year': 2012,
 'price': 599,
 'discounted_price': 479,
 'description': 'Red polo T-shirt, has a polo collar, short button placket, short sleeves',
 'image_url': 'http://assets.myntassets.com/v1/images/style/properties/778605c0c9603e03c84bf7f36d9756e4_images.jpg',
 'pattern': 'Solid',
 'fabric': 'Polyester',
 'sleeve_length': 'Short Sleeves',
 'occasion': 'Casual',
 'fit': 'Regular Fit',
 'neck': 'Polo Collar',
 'length': 'Regular'}

In [12]:
for key, value in sample_row.items():
    print(f"{key}: {value}")

id: 26233
product_display_name: Proline Men Red Polo T-shirt
brand_name: Proline
gender: Men
master_category: Apparel
sub_category: Topwear
article_type: Tshirts
base_colour: Red
season: Summer
usage: Casual
year: 2012
price: 599
discounted_price: 479
description: Red polo T-shirt, has a polo collar, short button placket, short sleeves
image_url: http://assets.myntassets.com/v1/images/style/properties/778605c0c9603e03c84bf7f36d9756e4_images.jpg
pattern: Solid
fabric: Polyester
sleeve_length: Short Sleeves
occasion: Casual
fit: Regular Fit
neck: Polo Collar
length: Regular


In [13]:
import importlib
!cd /kaggle/working/multi-modal-fashion-ecom && git pull

import clean_data
importlib.reload(clean_data)

Already up to date.


<module 'clean_data' from '/kaggle/working/multi-modal-fashion-ecom/data/scripts/clean_data.py'>

In [14]:
sample_search_text = clean_data.build_search_text(sample_row)

print(sample_search_text)

Proline Men Red Polo T-shirt Proline Apparel Topwear Tshirts Red Casual Solid Polyester Short Sleeves Casual Regular Fit Polo Collar Regular Red polo T-shirt, has a polo collar, short button placket, short sleeves


In [15]:
rows = [
    clean_data.build_row(record)
    for record in clean_records
]

products_df = pd.DataFrame(rows)

products_df["search_text"] = products_df.apply(
    lambda row: clean_data.build_search_text(row.to_dict()),
    axis=1
)

In [16]:
print("Rows:", len(products_df))
print("Columns:", len(products_df.columns))
print(products_df.columns.tolist())

Rows: 44424
Columns: 23
['id', 'product_display_name', 'brand_name', 'gender', 'master_category', 'sub_category', 'article_type', 'base_colour', 'season', 'usage', 'year', 'price', 'discounted_price', 'description', 'image_url', 'pattern', 'fabric', 'sleeve_length', 'occasion', 'fit', 'neck', 'length', 'search_text']


*Note: styleImages has multiple image-key variants; default is preferred, with front/left/top fallbacks.*

In [17]:
missing_image_rows = products_df[
    products_df["image_url"].isna()
].copy()

print("Rows missing image_url:", len(missing_image_rows))

Rows missing image_url: 0


In [18]:
products_df[
    products_df["id"].isin([12347, 39425, 39410, 39403, 39401])
][["id", "image_url"]]

,id,image_url
4178,12347,http://assets.myntassets.com/assets/images/pro...
7449,39425,http://assets.myntassets.com/v1/images/style/p...
8230,39410,http://assets.myntassets.com/v1/image/style/pr...
17492,39403,http://assets.myntassets.com/v1/images/style/p...
19251,39401,http://assets.myntassets.com/v1/images/style/p...


In [19]:
clean_data.run_sanity_checks(products_df)

All sanity checks passed.


## Some final inspections

In [20]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    44424 non-null  int64  
 1   product_display_name  44424 non-null  object 
 2   brand_name            44424 non-null  object 
 3   gender                44424 non-null  object 
 4   master_category       44424 non-null  object 
 5   sub_category          44424 non-null  object 
 6   article_type          44424 non-null  object 
 7   base_colour           44424 non-null  object 
 8   season                44403 non-null  object 
 9   usage                 44423 non-null  object 
 10  year                  44423 non-null  float64
 11  price                 44424 non-null  float64
 12  discounted_price      44424 non-null  float64
 13  description           44356 non-null  object 
 14  image_url             44424 non-null  object 
 15  pattern            

In [21]:
products_df.isna().sum().sort_values(ascending=False)

length                  40986
neck                    35085
fit                     34341
occasion                33424
sleeve_length           31591
fabric                  28835
pattern                 27927
description                68
season                     21
usage                       1
year                        1
brand_name                  0
product_display_name        0
id                          0
gender                      0
article_type                0
sub_category                0
master_category             0
base_colour                 0
image_url                   0
discounted_price            0
price                       0
search_text                 0
dtype: int64

In [22]:
for i, text in enumerate(
    products_df["search_text"].sample(5, random_state=42),
    start=1
):
    print(f"\n--- Sample {i} ---")
    print(text)


--- Sample 1 ---
Catwalk Women Pink & Copper Flats Catwalk Footwear Shoes Heels Pink Casual Casual Slip-on sandals have never looked this good. This pair from catwalk is versatile enough to go with everything: from jeans to ethnic wear to summer dresses . The colours make it a perfect choice for everyday wear. Upper Wide pink strap for the forefoot with a crystal embellishment Curved leather footbed in an ergonomic design for extra comfort A coloured patch on the footbed with embossed branding Outsole Plastic outsole with branded grooves Sandal Care Wipe surface with soft, clean cloth to remove dust Do not use a drier or store in direct sunlight Use a branded leather conditioner and brush to restore shoe shine

--- Sample 2 ---
W Women Purple Kurta W Apparel Topwear Kurtas Purple Ethnic Composition Purple floral print kurta made of 100% polyester, has three-quarter sleeves, scooped neck, stitch detailing on either side of the neckline and split side seams Fit Regular Wash care Machine

In [23]:
products_df["year"] = products_df["year"].astype("Int64")
products_df["year"].dtype

Int64Dtype()

In [24]:
clean_data.run_sanity_checks(products_df)

All sanity checks passed.


## Output 

In [25]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "products.parquet"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

products_df.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: /kaggle/working/multi-modal-fashion-ecom/data/processed/products.parquet


In [26]:
loaded_df = pd.read_parquet(OUTPUT_PATH)

print("Shape:", loaded_df.shape)
print("Year dtype:", loaded_df["year"].dtype)

Shape: (44424, 23)
Year dtype: Int64
